## 1. 分析目標

- 專案目標：探討跨星級飯店的評論分類模型,能否從評論內容預測飯店星級
- 訓練/測試資料：以 5 星、4 星、3 星、2 星飯店的 TripAdvisor 評論 (7:3 切分)
    - 四家飯店分別為各星級中評論數最多的一家

- 重點：
    - 模型能否從評論內容 (`content`) 正確分類出飯店星級 (`hotel_class`)
    - 不同星級飯店在評論內容與用字風格上的差異
    - 分析分類準確度與表現落差背後的原因

In [ ]:
import re
from pprint import pprint

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import jieba
from sklearn.model_selection import train_test_split, cross_validate, cross_val_predict, KFold
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
    RocCurveDisplay
)
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import LabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm

設定中文字體

In [ ]:
# 設定圖的中文字體 (Windows 使用 Microsoft JhengHei)
# 也可參考：https://pyecontech.com/2020/03/27/python_matplotlib_chinese/
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei'] #使圖中中文能正常顯示
plt.rcParams['axes.unicode_minus'] = False #使負號能夠顯示

## 2. 文字前處理
本次分析使用四個星級飯店的 TripAdvisor 評論資料(`5star.xlsx`、`4star.xlsx`、`3star.xlsx`、`2star.xlsx`)。
- 資料內容：各星級代表的飯店中評論數最多的那一家
- 原始欄位：reviewer, location, contributions, user_helpful_votes, date_posted, rating, title, content, stay_date, trip_type
- 本次分析主要使用 `content` 欄位作為輸入文字、`hotel_class` 作為分類標籤

先讀取資料並做基本前處理。

In [ ]:
# 讀取四個星級的飯店評論 (Excel 檔請先 pip install openpyxl)
df5 = pd.read_excel("raw_data2/5star.xlsx")
df4 = pd.read_excel("raw_data2/4star.xlsx")
df3 = pd.read_excel("raw_data2/3star.xlsx")
df2 = pd.read_excel("raw_data2/2star.xlsx")

# 加上 hotel_class 標記當作分類 label
df5["hotel_class"] = "5star"
df4["hotel_class"] = "4star"
df3["hotel_class"] = "3star"
df2["hotel_class"] = "2star"

# 四家全部合併 (對應原範本的 dcard)
dcard = pd.concat([df5, df4, df3, df2], ignore_index=True)

# 原範本 artUrl 是文章連結,這邊用 index 補上唯一 ID
dcard["artUrl"] = "review_" + dcard.index.astype(str)

dcard.head(3)

In [ ]:
# 看看有幾則評論
print(f"number of reviews: {dcard.shape[0]}")
print(f"date range: {(dcard['date_posted'].min(), dcard['date_posted'].max())}")
print(f"hotel_class distribution:\n{dcard['hotel_class'].value_counts()}")

+ 2.1 斷句
+ 2.2 斷詞（刪掉次數太少的、標點符號、停用字）

### 2.1 清理

利用問號、句號或驚嘆號等符號斷句，或是如果出現中文或是英文的省略號，像是`...`也會斷句，最後設定會去除結尾的空白符號。

In [ ]:
# 過濾 nan 的資料
dcard = dcard.dropna(subset=['content'])
dcard = dcard.dropna(subset=['hotel_class'])

# 移除網址格式
dcard["content"] = dcard["content"].astype(str).apply(
    lambda x: re.sub("(http|https)://\\S+", "", x)
)

# 只留下中文字
dcard["content"] = dcard["content"].apply(
    lambda x: re.sub("[^\u4e00-\u9fa5]+", "", x)
)

# 清完可能會有空字串
dcard = dcard[dcard["content"].str.len() > 0].reset_index(drop=True)
dcard.head()

本次分析只使用評論內文 `content` 作為文字分析對象,不使用 `title`。

In [ ]:
# 只留下會用到的欄位
dcard = dcard[["content", "artUrl", "hotel_class", "date_posted"]]
dcard.head()

In [ ]:
# 看看有幾篇文章
print(f"total docs: {dcard.shape[0]}")


### 2.2 斷詞

In [ ]:
# 設定繁體中文詞庫
jieba.set_dictionary("./dict/dict.txt")

# 新增stopwords
# jieba.analyse.set_stop_words('./dict/stopwords.txt') #jieba.analyse.extract_tags才會作用
with open("./dict/stopwords.txt", encoding="utf-8") as f:
    stopWords = [line.strip() for line in f.readlines()]

In [ ]:
# 設定斷詞 function
def getToken(row):
    seg_list = jieba.cut(row, cut_all=False)
    seg_list = [
        w for w in seg_list if w not in stopWords and len(w) > 1
    ]  # 篩選掉停用字與字元數大於1的詞彙
    return seg_list

In [ ]:
dcard["words"] = dcard["content"].apply(getToken).map(" ".join)
dcard.head()

### 2.3 資料集基本檢視

檢視資料內容

In [ ]:
print(f"total reviews: {len(dcard['artUrl'].unique())}")
print(f"hotel_class distribution:\n{dcard['hotel_class'].value_counts()}")

## 3. 分類模型的訓練流程
### 3.1 根據7:3的比例切分資料集
利用 sklearn 中的 train_test_split 函數將 `raw_data` 隨機切成 7:3，設置 random_state 讓每次切分的結果一致。`y_train`和`y_test`分別為訓練資料和測試資料的預測目標。

In [ ]:
data = dcard
X = data["words"]
y = data["hotel_class"]

# 把整個資料集七三切
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=777, stratify=y
)

print(X_train.head())
print(y_train.head())

In [ ]:
# 看一下各個資料集切分的比例,應該要一致
print(
    f"raw data percentage :\n{data['hotel_class'].value_counts(normalize=True) * 100}"
)
print(f"\ntrain percentage :\n{y_train.value_counts(normalize=True) * 100}")
print(f"\ntest percentage :\n{y_test.value_counts(normalize=True) * 100}")

### 3.2 將文章轉為 DTM

DTM(document term matrix) :
+ 將不同的文章 (document) 以文章中出現過的字詞(term)表示
    + row 是document (文件)
    + column 是字詞 (term)
    + row 內的數字是出現的字數

DTM裡面的值可以有不同的表示方法
+ (1) 依據詞頻 (classic BoW)
    + 用 `CountVectorizer()`
    + unigrams and bigrams
    + [sklearn.feature_extraction.text.CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
+ (2) 依據tfidf (advanced variant of BoW)
    + 篩選出現次數大於10的字
    + 用 `TfidfVectorizer()`
    + [sklearn.feature_extraction.text.TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
+ 常用參數介紹
    + max_features: 取 詞頻 / tfidf 前多少的字

### 3.3 套入正式的資料集

模型訓練範例: logistic regression + cv tokenizer

In [ ]:
vectorizer = CountVectorizer(max_features=1000)
print(vectorizer)

In [ ]:
X_train.head()

In [ ]:
vec_train = vectorizer.fit_transform(X_train)
vocabulary = vectorizer.get_feature_names_out()

Count_df = pd.DataFrame(columns = vocabulary, data = vec_train.toarray())
Count_df

In [ ]:
# 不需重新 `fit()` data，因前面已經 `fit()` 過了
# 只需將測試數據使用之前訓練好的 vectorizer 轉換為相同的特徵表示形式，而不需要重新fit。
# 如果對測試數據再次 fit vectorizer，可能會導致使用了測試數據的信息，進而導致模型的不穩定性和過度擬合的問題。
vec_test = vectorizer.transform(X_test)
print(vec_train.shape)
print(vec_test.shape)

In [ ]:
# 建立分類器模型
clf = LogisticRegression(max_iter=1000)
clf.fit(vec_train, y_train)
clf

In [ ]:
clf.classes_

使用train set訓練完後，用測試集試試看模型的分類結果

In [ ]:
y_pred = clf.predict(vec_test)
y_pred_proba = clf.predict_proba(vec_test)
print(y_pred[:10])

觀察看看模型輸出的類別機率

In [ ]:
print(y_pred_proba.shape)
y_pred_proba[0,:]

每一列代表一則評論對各星級類別 (2star/3star/4star/5star) 的預測機率,加總為 1。模型會選機率最高的類別作為預測結果。

### 3.4 模型評估

In [ ]:
## Accuracy, Precision, Recall, F1-score
print(classification_report(y_test, y_pred))

輸出混淆矩陣

In [ ]:
classes = clf.classes_
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
## Plot confusion matrix
fig, ax = plt.subplots()
sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap=plt.cm.Blues, cbar=False)
ax.set(
    xlabel="Pred",
    ylabel="True",
    xticklabels=classes,
    yticklabels=classes,
    title="Confusion matrix",
)
plt.yticks(rotation=0)

### 3.4 TF-IDF

改使用 TF-IDF 的 DTM 來代表文章，訓練分類模型的效果。

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000)
vec_train = vectorizer.fit_transform(X_train)
vec_test = vectorizer.transform(X_test)
vocabulary = vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(columns = vocabulary, data = vec_train.toarray())
tfidf_df

In [ ]:
clf.fit(vec_train, y_train)
y_pred = clf.predict(vec_test)
y_pred_proba = clf.predict_proba(vec_test)

# results
## Accuracy, Precision, Recall, F1-score
print(classification_report(y_test, y_pred))

效果有好一點點，但沒有差很多。

### 3.5 CV

cross_validate：

- 返回多種指標的結果，包括 f1_macro、recall_macro 和 precision_macro，並能返回每折訓練和測試的時間、模型等詳細信息。

- 更適合用於綜合評估模型的性能並返回模型本身。

In [ ]:
clf = LogisticRegression()
vec_train = TfidfVectorizer(max_features=1000).fit_transform(X_train)

scores = cross_validate(clf, vec_train, y_train, cv=5, scoring=("f1_macro", "recall_macro", "precision_macro"), return_estimator=True)
pprint(scores)

cross_val_predict：

- 返回每一折的預測標籤，用來後續生成分類報告（classification_report）。

- 適用於需要比較預測結果的情況。

In [ ]:
y_pred = cross_val_predict(clf, vec_train, y_train, cv=5)
print(classification_report(y_train, y_pred))

## 4. 比較不同模型效果

In [ ]:
# 定義模型訓練組合
## pipeline: 資料處理 vectorizer + 分類器 clf
## 由於 cross-validation 會自動將資料分成 train/test,因此 input 只要給 X, y 即可

def train_cv(vectorizer, clf, X, y):
    ## train classifier
    vec_X = vectorizer.fit_transform(X).toarray()

    ## get cv results
    cv_results = cross_validate(clf, vec_X, y, cv=5, return_estimator=True)
    y_pred = cross_val_predict(clf, vec_X, y, cv=5)
    # classification_report 回傳 dict 方便後續比較
    report = classification_report(y, y_pred, output_dict=True, zero_division=0)
    return report

因為TF-IDF的效果好一點，所以我們選擇使用TF-IDF來實作後面的模型。

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000)
clf = LogisticRegression()
result = train_cv(vectorizer, clf, X_train, y_train)

輸出：LogisticRegression、DecisionTree、SVC、RandomForest的各項檢測指標

In [ ]:
# 準備訓練資料
X = data["words"]
y = data["hotel_class"]

# 把整個資料集七三切
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=777, stratify=y
)
# 定義模型訓練組合
model_set = dict()
model_set['clf_logistic'] = LogisticRegression(max_iter=1000)
model_set['clf_dtree'] = DecisionTreeClassifier(random_state=777)
model_set['clf_svm'] = svm.SVC(probability=True) # 要使用SVM的predict_proba的話,必須在叫出SVC的時候就將probability設為True
model_set['clf_rf'] = RandomForestClassifier(random_state=777)

# 用 TF-IDF 作為特徵,跑每個模型的 CV
vectorizer = TfidfVectorizer(max_features=1000)
result_set = dict()
for name, clf in model_set.items():
    print(f"--- training {name} ---")
    result_set[name] = train_cv(vectorizer, clf, X_train, y_train)

分別觀察各個分類模型在不同類別的評估指標表現如何

In [ ]:
result_set['clf_logistic']

找出f1-score表現最好的模型是哪個，作為我們最終得到的分類器

In [ ]:
max = 0
best_model_name = ""
best_model_metric = "f1-score"

## choose max f1-score model from result_set
for k, v in result_set.items():
    if v['weighted avg'][best_model_metric] > max:
        max = v['weighted avg'][best_model_metric]
        best_model_name = k
print(f"best model: {best_model_name}")
pprint(result_set[best_model_name])

從結果可以看到LogisticRegression為我們的best model

In [ ]:
y_pred = model_set['clf_logistic'].predict(vectorizer.transform(X_test).toarray())
print(classification_report(y_test, y_pred))

輸入資料測試模型的效果

In [ ]:
# 用一則範例評論測試看看
test_review = "房間 乾淨 服務 態度 親切 早餐 豐盛 推薦"
model_set['clf_logistic'].predict(vectorizer.transform([test_review]).toarray())

## 5. 分析可解釋模型的結果

### 5.1 各字詞特徵的迴歸係數

因為LogisticRegression為我們的best model，所以我們用該模型結合cv tokenizer實作下面的範例

In [ ]:
def plot_coef(logistic_reg_model, feature_names, top_n=10):
    # 選出某個類別的前10大影響力字詞
    log_odds = logistic_reg_model.coef_.T
    coef_df = pd.DataFrame(
        log_odds, 
        columns=logistic_reg_model.classes_, index=feature_names
    )
    for label in coef_df.columns:
        select_words = (
            coef_df[[label]]
            .sort_values(by=label, ascending=False)
            .iloc[np.r_[0:top_n, -top_n:0]]
        )
        word = select_words.index
        count = select_words[label]
        category_colors = np.where(
            select_words[label] >= 0, "darkseagreen", "rosybrown"
        )  # 設定顏色

        fig, ax = plt.subplots(figsize=(8, top_n*0.8))  # 設定畫布
        plt.rcParams["axes.unicode_minus"] = False

        ax.barh(word, count, color=category_colors)
        ax.invert_yaxis()
        ax.set_title(
            "Coeff increase/decrease odds ratio of 「" + label + "」 label the most",
            loc="left",
            size=16,
        )
        ax.set_ylabel("word", size=14)
        ax.set_xlabel("odds ratio", size=14)

從下面的圖表可以了解每個星級類別對應的關鍵字,
像是 5 星飯店的評論可能出現「奢華」、「貴賓」、「高級」等詞,
而 2 星飯店的評論可能出現「便宜」、「簡陋」、「普通」等詞。

In [ ]:
# 確保 vectorizer 跟 clf_logistic 使用同一份訓練資料配套
vectorizer = TfidfVectorizer(max_features=1000)
vec_train = vectorizer.fit_transform(X_train)
model_set['clf_logistic'] = LogisticRegression(max_iter=1000)
model_set['clf_logistic'].fit(vec_train, y_train)

plot_coef(
    logistic_reg_model=model_set['clf_logistic'],
    feature_names=vectorizer.get_feature_names_out(),
    top_n=10
)

## 6. 用訓練好的分類器檢視各星級的詳細分類表現

前面我們訓練好了分類器,這邊進一步觀察模型對各星級飯店的分類細節。
特別是看模型在哪些星級分類最準、哪些容易混淆,以及錯誤分類的樣本長什麼樣。

In [ ]:
# 不需要另外讀資料,直接用前面切分好的 test set
ct = pd.DataFrame({
    'content': X_test.apply(lambda _: ''),   # 先佔位,下面再把原 content 補回
    'words': X_test.values,
    'hotel_class': y_test.values,
    'artUrl': ['test_' + str(i) for i in range(len(X_test))]
}, index=X_test.index)

# 把原始 content 補回來 (用 index 對應)
ct['content'] = data.loc[X_test.index, 'content'].values

ct.head()

In [ ]:
# test set 已經清理過,這邊直接確認大小
print(f"test set 共 {len(ct)} 則評論")
ct.head()

觀察一下資料集的分佈狀況

In [ ]:
ct['hotel_class'].value_counts()

直接用 best model 預測 test set 的 hotel_class

In [ ]:
X = ct['words']
y = ct['hotel_class']

y_pred = model_set[best_model_name].predict(vectorizer.transform(X).toarray())
print(classification_report(y, y_pred, zero_division=0))

從 classification report 可以觀察:
- 哪個星級的飯店最容易被正確分類 (通常是極端的 5 星和 2 星)
- 哪些星級之間容易混淆 (例如 4 星和 5 星的評論語言風格可能很像)
- 各星級的 recall 和 precision 差異

In [ ]:
ct['pred'] = y_pred
ct.loc[:, ['words', 'hotel_class', 'pred']]

將錯誤分類的結果篩選出來

In [ ]:
false_pred = ct.query("hotel_class != pred").loc[:, ['words', 'hotel_class', 'pred']]
false_pred

觀察看看 2 星飯店的評論 (最低星級),模型是否會誤判成其他星級

In [ ]:
false_pred.loc[false_pred['hotel_class'] == '2star', :]

挑選一則被誤判的 2 星評論觀察看看,分析為何會被預測成其他星級

In [ ]:
# 抓第一筆 2star 被誤判的評論 words
low_mis = false_pred.loc[false_pred['hotel_class'] == '2star']
if len(low_mis) > 0:
    idx = low_mis.index[0]
    pprint(low_mis.loc[idx, 'words'])
else:
    print("沒有 2star 的誤判樣本")

In [ ]:
# 看原始評論內容
if len(low_mis) > 0:
    idx = low_mis.index[0]
    print(ct.loc[idx, 'content'])

## 主題模型
主題模型（Topic Modeling）是一種無監督學習技術，用於從大量文本中自動識別潛在的主題。本次使用LDA模型進行主題分析，並結合視覺化工具與情緒分析，探索文本資料中的主題分佈與情緒趨勢。

In [ ]:
import time 
from functools import reduce
from collections import Counter
from pprint import pprint
from wordcloud import WordCloud 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jieba
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
from gensim.models.ldamulticore import LdaMulticore
from gensim.matutils import corpus2csc, corpus2dense, Sparse2Corpus

import pyLDAvis
import pyLDAvis.gensim_models

In [ ]:
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

In [ ]:
# 設定中文字體
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei'] #使圖中中文能正常顯示
plt.rcParams['axes.unicode_minus'] = False #使負號能夠顯示

In [ ]:
# 重新讀四家飯店做 LDA (無監督學習,用全部資料效果更好)
df5 = pd.read_excel("./raw_data2/5star.xlsx"); df5["hotel_class"] = "5star"
df4 = pd.read_excel("./raw_data2/4star.xlsx"); df4["hotel_class"] = "4star"
df3 = pd.read_excel("./raw_data2/3star.xlsx"); df3["hotel_class"] = "3star"
df2 = pd.read_excel("./raw_data2/2star.xlsx"); df2["hotel_class"] = "2star"
dcard = pd.concat([df5, df4, df3, df2], ignore_index=True)
dcard["artUrl"] = "all_" + dcard.index.astype(str)
dcard.head(3)

In [ ]:
dcard = dcard.dropna(subset=['content', 'hotel_class'])
# 移除網址、只留下中文字
dcard["content"] = dcard["content"].astype(str).str.replace("(http|https)://\\S+", "", regex=True)
dcard["content"] = dcard["content"].str.replace("[^\u4e00-\u9fa5]+", "", regex=True)
dcard = dcard[dcard["content"].str.len() > 0].reset_index(drop=True)

In [ ]:
# 設定繁體中文詞庫
jieba.set_dictionary("./dict/dict.txt")

# 新增stopwords
with open("./dict/stopwords.txt", encoding="utf-8") as f:
    stopWords = [line.strip() for line in f.readlines()]

# 設定斷詞 function
def getToken(row):
    seg_list = jieba.cut(row, cut_all=False)
    seg_list = [w for w in seg_list if w not in stopWords and len(w) > 1]
    return seg_list

dcard["words"] = dcard["content"].apply(getToken)
dcard.head()

將斷詞後的doc['words']轉換成list

In [ ]:
docs = dcard['words'].to_list()
docs[0]

建立並過濾詞彙表（dictionary），只保留特定條件的詞彙

    no_below=5 出現在少於 5 篇文章中的詞會被移除
    
    no_above=0.99 出現在超過 99% 文件中的詞會被移除

In [ ]:
dictionary = Dictionary(docs)

dictionary.filter_extremes(no_below=5, no_above=0.99)
print(dictionary)

將斷詞結果建構語料庫(corpus)之後，利用語料庫把每篇文章數字化。
<br>每個詞彙都被賦予一個 ID 及頻率(word_id，word_frequency)。<br>

舉例來說：<br>
第一篇文章數字化結果為：corpus[600]:[(2, 2), (6, 1), (20, 2), .... ]，element 為文章中每個詞彙的 id 和頻率。<br>
代表：'世界'出現2次、'之戰'出現一次...以此類推

In [ ]:
for idx, (k, v) in enumerate(dictionary.token2id.items()):
    print(f"{k}: {v}")
    if idx > 10:
        break

In [ ]:
pprint(" ".join(dcard['words'].iloc[0]))

In [ ]:
dictionary.doc2bow(dcard['words'].iloc[0])[:10]

將docs轉換成BOW形式<br>
把每篇文件的 token list 轉換成一組 (token_id, count) 的 list

In [ ]:
# 建立 Bag-of-words 作為文章的特徵表示
# 用 gensim ldamodel input 需要將文章轉換成 bag of words 
corpus = [dictionary.doc2bow(doc) for doc in docs]

In [ ]:
ldamodel = LdaModel(
    corpus=corpus, 
    id2word=dictionary, # 字典
    num_topics=10, # 生成幾個主題數
    random_state=2024, # 亂數
)

In [ ]:
ldamodel.print_topics()

In [ ]:
ldamodel.get_document_topics(corpus[0])

### 查看 LDA 模型指標

In [ ]:
# perplexity
perplexity = ldamodel.log_perplexity(corpus)
perplexity

In [ ]:
np.exp2(-perplexity)

In [ ]:
# npmi
NPMI_model_lda = CoherenceModel(model=ldamodel, texts=docs, coherence='c_npmi')
NPMI_lda = NPMI_model_lda.get_coherence()
print('這個主題模型的 PMI score: ', NPMI_lda)

In [ ]:
NPMI_model_lda.get_coherence_per_topic()

### 透過指標找出最佳主題數

In [ ]:
t0 = time.time()

topic_num_list = np.arange(2, 8)
result = {"topic_num":[], "perplexity":[], "pmi":[]}
model_set = dict()


for topic_num in topic_num_list:
    # perplexity
    model = LdaModel(
        corpus = corpus,
        num_topics = topic_num ,
        id2word=dictionary,
        random_state = 1500,
        passes=5 # 訓練次數
        )
    
    loss = model.log_perplexity(corpus)
    pmi = CoherenceModel(model=model, texts=docs, coherence='c_npmi').get_coherence()
    perplexity = np.exp(-1. * loss)
    
    # model_set[f'k_{topic_num}'] = model
    
    result['topic_num'].append(topic_num)
    result['perplexity'].append(perplexity)
    result['pmi'].append(pmi)
    
print(f"花費時間: {time.time() - t0} sec")

In [ ]:
result = pd.DataFrame(result)
result

In [ ]:
result.plot.line(x='topic_num', y='perplexity')

In [ ]:
result.plot.line(x='topic_num', y='pmi')

綜合考量模型的解釋力與主題的語意一致性，雖然在 Perplexity 指標中 7 題的數值最低，代表對語料擬合最佳，但在 PMI 指標中，6 題表現相對較佳，顯示其主題內部詞彙的語意較集中、一致性較好。為兼顧模型效果與主題可解釋性，最終決定採用 6 個主題 作為 LDA 模型的主題數設定。

### 視覺化呈現

In [ ]:
best_model = LdaModel(
    corpus = corpus,
    num_topics = 6,
    id2word=dictionary,
    random_state = 1500,
    passes = 5 # 訓練次數
    )

In [ ]:
import gensim
gensim.__version__

In [ ]:
from pyLDAvis import gensim_models

In [ ]:
pyLDAvis.enable_notebook()
p = pyLDAvis.gensim_models.prepare(best_model, corpus, dictionary)
p

目前看起來分類結果在主題數為6還不錯，主題之間沒有重疊，大小還算平均。

In [ ]:
pyLDAvis.save_html(p, "lda_tripadvisor.html")

## GuidedLDA

In [ ]:
import guidedlda

In [ ]:
word2id = dictionary.token2id

In [ ]:
# 根據飯店評論常見主題設定種子詞
seed_topic_list = [
    ["房間", "床", "浴室", "空間", "設備"],      # 房間設施
    ["服務", "態度", "人員", "櫃台", "親切"],    # 服務與接待
    ["早餐", "餐廳", "飲料", "食物", "菜色"],    # 餐飲
    ["地點", "交通", "位置", "捷運", "車站"],    # 地點與交通
    ["價格", "便宜", "值得", "划算", "推薦"],    # 價格/CP 值
    ["乾淨", "整潔", "舒適", "安靜", "衛生"],    # 環境品質
]

In [ ]:
# 將 seed words 對應到 dictionary 的 token id
# 有些詞可能因 filter_extremes 被移除,需要 skip
seed_topics = {}
missing = []
for t_id, st in enumerate(seed_topic_list):
    for word in st:
        if word in word2id:
            seed_topics[word2id[word]] = t_id
        else:
            missing.append(word)
print(f"有效 seed words: {len(seed_topics)} 個")
if missing:
    print(f"這些詞不在 dictionary (可能被 filter 移除): {missing}")

In [ ]:
# (此格跟上格作用相同,保留原範本結構)
seed_topics = {}
for t_id, st in enumerate(seed_topic_list):
    for word in st:
        if word in word2id:
            seed_topics[word2id[word]] = t_id

In [ ]:
# guidedlda 需要 DTM 格式作為 input，因此這邊利用 corpus2dense() 方法進行轉換
X = corpus2dense(corpus, len(dictionary), len(corpus)).T.astype(np.int64)

In [ ]:
model = guidedlda.GuidedLDA(n_topics=6, n_iter=100, random_state=7, refresh=20)
model.fit(X, seed_topics=seed_topics, seed_confidence=1)

In [ ]:
# 整理／顯示主題模型結果
n_top_words = 10
topic_word = model.topic_word_
# 取得corpus全部的詞彙表
vocab = tuple(dictionary.token2id.keys())

for i, topic_dist in enumerate(topic_word):
    # 依照詞語機率從小到大排序，找出每個主題的前十個關鍵詞
    topic_words = np.array(vocab)[np.argsort(topic_dist)][: -(n_top_words + 1) : -1]
    print("Topic {}: {}".format(i, " ".join(topic_words)))
    

doc_topic = model.doc_topic_ # 文件-主題 分佈
term_freq = tuple(dictionary.cfs.values()) # 每個詞在整個語料中出現的總次數
doc_len = [sum(v for k, v in doc) for doc in corpus] # 每篇文章的長度

## LDAvis
pyLDAvis.enable_notebook()
p = pyLDAvis.prepare(topic_word, doc_topic, doc_len, vocab = vocab, term_frequency = term_freq)
p

根據輸出的各主題 top words,可以觀察是否符合預設的 6 大主題
(房間設施 / 服務 / 餐飲 / 地點 / 價格 / 環境品質)。

接下來擴充每組的 seed words,希望讓模型更穩定地收斂到這些語意方向。

In [ ]:
seed_topic_list = [
    # 房間設施
    ["房間", "床", "浴室", "空間", "設備", "電視", "冷氣", "淋浴", "冰箱", "沙發", "窗戶", "房型"],

    # 服務與接待
    ["服務", "態度", "人員", "櫃台", "親切", "熱心", "員工", "接待", "笑容", "專業", "客房", "服務員"],

    # 餐飲
    ["早餐", "餐廳", "飲料", "食物", "菜色", "餐點", "咖啡", "晚餐", "料理", "美味", "好吃", "自助"],

    # 地點與交通
    ["地點", "交通", "位置", "捷運", "車站", "機場", "步行", "附近", "方便", "距離", "便利", "商圈"],

    # 價格/CP 值
    ["價格", "便宜", "值得", "划算", "推薦", "費用", "合理", "超值", "物超所值"],

    # 環境品質
    ["乾淨", "整潔", "舒適", "安靜", "衛生", "明亮", "寬敞", "漂亮", "高級", "豪華", "老舊", "陳舊"],
]

# 重新建 seed_topics
seed_topics = {}
for t_id, st in enumerate(seed_topic_list):
    for word in st:
        if word in word2id:
            seed_topics[word2id[word]] = t_id

In [ ]:
model = guidedlda.GuidedLDA(n_topics=6, n_iter=100, random_state=7, refresh=20)
model.fit(X, seed_topics=seed_topics, seed_confidence=1)

In [ ]:
n_top_words = 10
topic_word = model.topic_word_

for i, topic_dist in enumerate(topic_word):
    topic_words = np.array(vocab)[np.argsort(topic_dist)][: -(n_top_words + 1) : -1]
    print("Topic {}: {}".format(i, " ".join(topic_words)))
    

doc_topic = model.doc_topic_
term_freq = tuple(dictionary.cfs.values())
doc_len = [sum(v for k, v in doc) for doc in corpus]

## LDAvis
pyLDAvis.enable_notebook()
p = pyLDAvis.prepare(topic_word, doc_topic, doc_len, vocab = vocab, term_frequency = term_freq)
p

從 PyLDAVis 的視覺化結果來看，各主題間大致有明顯區隔，且高頻詞與種子詞匹配良好，顯示模型已能穩定辨識語料中的潛在主題結構

## 主題分佈的應用，搭配其他文章資訊 (LDA)

In [ ]:
# 取得每條新聞的主題分佈
topics_doc = best_model.get_document_topics(corpus)

In [ ]:
topics_doc[0]

#### 將 LDA 模型推論後的每篇文件的 主題分布（也就是 𝜃）轉換成一個 NumPy 矩陣（array）

In [ ]:
# 把 gensim 的稀疏表示法轉成稀疏矩陣
m_theta = corpus2csc(topics_doc).T.toarray() # 倒置讓shape變為(num_docs, num_topics)
m_theta

In [ ]:
# 將主題的機率分布轉換成主題標籤
dcard['topic_label'] = m_theta.argmax(axis=1) + 1

#### 統計一下各個主題的數量

In [ ]:
dcard['topic_label'].value_counts()

In [ ]:
dcard.head()

#### 查看每個飯店星級下的主題分布

In [ ]:
dcard.groupby('hotel_class')['topic_label'].value_counts(normalize=True)

In [ ]:
hotel_topic = dcard.groupby('hotel_class')['topic_label'].value_counts(normalize=True).unstack()
hotel_topic

#### 視覺化呈現各星級飯店的主題分布

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
hotel_topic.plot.bar(ax=ax, stacked=True, color=plt.cm.Set3.colors)
ax.legend(loc='lower right', title='Topic')
ax.set_title("各飯店星級的主題分布")
ax.set_ylabel("比例")
plt.xticks(rotation=0)
plt.tight_layout()

觀察重點:
- 哪些主題在低星級飯店 (2 星) 比例特別高 → 可能是客訴熱點
- 哪些主題在高星級飯店 (5 星) 比例特別高 → 可能是讚賞的重點

下面我們挑出在低分評論 (rating <= 2) 中最常出現的主題來做深入分析。

In [ ]:
import matplotlib.pyplot as plt

# 找出 2 星飯店評論中最常出現的主題
low_star_docs = dcard[dcard['hotel_class'] == '2star']
low_star_top_topic = low_star_docs['topic_label'].value_counts().idxmax()
print(f"2 星飯店評論最常出現的主題是: Topic {low_star_top_topic}")

# 看這個主題在各星級的評論數量分布
topic_by_class = dcard[dcard['topic_label'] == low_star_top_topic]['hotel_class'].value_counts().reindex(['5star','4star','3star','2star'])

plt.figure(figsize=(8, 5))
topic_by_class.plot(kind='bar', color='skyblue')
plt.title(f"Topic {low_star_top_topic} 在各星級飯店的評論數量")
plt.xlabel("hotel_class")
plt.ylabel("評論數")
plt.xticks(rotation=0)
plt.tight_layout()

從圖中可以看到此主題在各星級的分布:
- 若主要集中在 2 星 → 是低星級飯店評論的特色主題 (可能是客訴熱點)
- 若各星級都有不少 → 此主題是飯店評論的通用話題

接下來針對 2 星飯店的評論進行情緒分析。

## 了解 2 星飯店評論集中在哪些主題

In [ ]:
# 建立 2 星飯店評論的 subset (沿用原變數名 dcard_lable_6 方便後面程式不用改)
dcard_lable_6 = dcard[dcard['hotel_class'] == '2star'].copy()

print("2 星飯店評論共 ", len(dcard_lable_6), "則")
print("\n主題分布:")
print(dcard_lable_6["topic_label"].value_counts().sort_index())

(原範本此處有 Dcard 研究所日程表圖片,飯店情境不適用,已略過)

在飯店評論情境下,2 星飯店評論的集中點通常會是:
- **環境品質** (髒、吵、老舊)
- **房間設施** (冷氣不冷、熱水不熱、壞掉)
- **服務態度** (櫃台冷淡、不處理問題)

下面透過情緒分析觀察 2 星飯店評論中具體的負面情緒詞。

我們針對 2 星飯店的評論做情緒分析,主要是因為這類評論最可能包含客訴內容與負面情緒表達。
透過 LIWC 中文情緒字典,可以具體看出哪些負面詞彙最常出現,
進一步歸納出客人最常抱怨的點,作為飯店改善的參考。

#### 結合情緒分類
針對 2 星飯店評論做正負向情緒分析

In [ ]:
# 讀取情緒字典
liwc_dict = pd.read_csv("./dict/liwc/LIWC_CH.csv")
liwc_dict = liwc_dict.rename(columns={'name': 'word', "class": 'sentiments'})
liwc_dict.head()

In [ ]:
dcard_lable_6 = dcard_lable_6.explode("words").rename(columns={"words": "word"})

In [ ]:
dcard_liwc_df = pd.merge(
    dcard_lable_6[["artUrl", "date_posted", "content", "word"]],
    liwc_dict, how="left"
)
dcard_liwc_df.head()

In [ ]:
sentiment_count = pd.DataFrame(
    dcard_liwc_df.groupby(["date_posted", "sentiments"]).size()
).reset_index()

mask = (sentiment_count['sentiments'] == "positive") | (sentiment_count['sentiments'] == "negative")
sentiment_count = sentiment_count.loc[mask]

sentiment_count = sentiment_count.rename(columns={0: "size"})
sentiment_count = sentiment_count.sort_values(["date_posted"])
sentiment_count

#### 正負向情緒詞彙頻率折線圖圖

In [ ]:
colors = ["tab:blue", "tab:orange"]
pos = sentiment_count[sentiment_count["sentiments"] == "positive"]
neg = sentiment_count[sentiment_count["sentiments"] == "negative"]

fig, ax = plt.subplots(figsize=(10, 5))

rolling_days = 14
ax.plot(pos["date_posted"], pos['size'].rolling(rolling_days).mean(), color=colors[0], label='positive')
ax.plot(neg["date_posted"], neg["size"].rolling(rolling_days).mean(), color=colors[1], label='negative')

plt.xlabel("date_posted")
plt.ylabel("詞頻 (14 天 rolling mean)")
ax.legend(loc="lower left")
fig.autofmt_xdate()
plt.title("低分評論的正負向情緒詞頻率趨勢")
plt.tight_layout()

In [ ]:
# 與 apply 不同,transform 返回的結果與原數據的形狀一致
sentiment_count = sentiment_count.assign(
    ratio=sentiment_count.groupby("date_posted")["size"].transform(lambda n: n / n.sum())
)
sentiment_count.head(10)

In [ ]:
pos = sentiment_count[sentiment_count["sentiments"] == "positive"]
neg = sentiment_count[sentiment_count["sentiments"] == "negative"]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

rolling_days = 14
ax.plot(pos["date_posted"], pos["ratio"].rolling(rolling_days).mean(), color=colors[0], label='positive')
ax.plot(neg["date_posted"], neg["ratio"].rolling(rolling_days).mean(), color=colors[1], label='negative')

plt.xlabel("date_posted")
plt.ylabel("ratio (14 天 rolling mean)")
ax.legend(loc="lower left")
fig.autofmt_xdate()
plt.title("低分評論的正負情緒比例折線圖")
plt.tight_layout()

#### 文章為單位的情緒分析

In [ ]:
sentiment_count_by_article = pd.DataFrame(
    dcard_liwc_df.groupby(["artUrl", "sentiments"]).size()
).reset_index()
sentiment_count_by_article = sentiment_count_by_article.rename(columns={0: "size"})
sentiment_count_by_article.head(10)

In [ ]:
dcard_sentiment_value_by_article = (
    sentiment_count_by_article.pivot_table(
        index="artUrl", columns="sentiments", values="size", fill_value=0
    )
    .reset_index()
    .rename_axis(None, axis=1)
)
dcard_sentiment_value_by_article.head()

In [ ]:
# sentiment 計算方式: positive - negative
dcard_sentiment_value_by_article["sentiment_value"] = (
    dcard_sentiment_value_by_article["positive"]
    - dcard_sentiment_value_by_article["negative"]
)
dcard_sentiment_value_by_article.head()

In [ ]:
dcard_sentiment_value_by_article['sentiment_class'] = dcard_sentiment_value_by_article['sentiment_value'].apply(lambda x: "正向" if x > 0 else "負向" )
dcard_sentiment_value_by_article.head(10)

In [ ]:
clear_df_sentiment = pd.merge(
    dcard_lable_6,
    dcard_sentiment_value_by_article[['artUrl', 'sentiment_class']],
    how="left"
)
clear_df_sentiment["date_posted"] = pd.to_datetime(clear_df_sentiment["date_posted"], errors='coerce')
clear_df_sentiment['date_posted'] = clear_df_sentiment['date_posted'].dt.date

clear_df_sentiment.head()

In [ ]:
sentiment_art_count = pd.DataFrame(
    clear_df_sentiment.groupby(["date_posted", "sentiment_class"]).size()
).reset_index()
sentiment_art_count = sentiment_art_count.rename(columns={0: "size"})
sentiment_art_count = sentiment_art_count.sort_values(["date_posted"])
sentiment_art_count

In [ ]:
colors = ["tab:blue", "tab:orange"]
pos = sentiment_art_count[sentiment_art_count["sentiment_class"] == "正向"]
neg = sentiment_art_count[sentiment_art_count["sentiment_class"] == "負向"]

fig, ax = plt.subplots(figsize=(10, 5))

rolling_days = 14
ax.plot(pos["date_posted"], pos['size'].rolling(rolling_days).mean(), color=colors[0], label='正向')
ax.plot(neg["date_posted"], neg['size'].rolling(rolling_days).mean(), color=colors[1], label='負向')

plt.xlabel("date_posted")
plt.ylabel("評論數 (14 天 rolling mean)")
ax.legend(loc="lower left")
fig.autofmt_xdate()
plt.title("低分評論中正向/負向評論數量趨勢")
plt.tight_layout()

#### 2 星飯店評論的負向情緒文字雲

In [ ]:
mask = clear_df_sentiment['sentiment_class'] == "負向"
ptt_df_wc = clear_df_sentiment.loc[mask]
ptt_df_wc.head(10)

In [ ]:
#clear_df_sentiment_wc = clear_df_sentiment.explode("words").rename(columns={"words": "word"})
#clear_df_sentiment_wc.head(3)
clear_df_sentiment_wc = ptt_df_wc

In [ ]:
mask = clear_df_sentiment_wc['sentiment_class'] == "負向"
dcard_df_wc = clear_df_sentiment_wc.loc[mask, ["date_posted", "word"]]

word_count_count = pd.DataFrame(
    dcard_df_wc.groupby(["word"]).size()
).reset_index().rename(columns={0: "size"})
word_count_count = word_count_count.sort_values("size", ascending=False)
word_count_count.head(20)

In [ ]:
# wordcloud 的 input 是 dictionary
font_path = './raw_data/SourceHanSansTW-Regular.otf'  # 中文字型路徑
wc_dict = dict(zip(word_count_count['word'], word_count_count['size']))
cloud = WordCloud(scale=4, max_words=200, background_color="white", font_path=font_path)
cloud.generate_from_frequencies(wc_dict)

plt.figure(figsize=(8, 4), dpi=150)
plt.imshow(cloud, interpolation="bilinear")
plt.axis("off")
plt.title("2 星飯店評論 — 負向情緒文字雲")
plt.show()

In [ ]:
mask = clear_df_sentiment['sentiment_class'] == "正向"
ptt_df_wc = clear_df_sentiment.loc[mask]
ptt_df_wc.head(10)

In [ ]:
clear_df_sentiment_wc = ptt_df_wc

In [ ]:
mask = clear_df_sentiment_wc['sentiment_class'] == "正向"
dcard_df_wc = clear_df_sentiment_wc.loc[mask, ["date_posted", "word"]]

word_count_count = pd.DataFrame(
    dcard_df_wc.groupby(["word"]).size()
).reset_index().rename(columns={0: "size"})
word_count_count = word_count_count.sort_values("size", ascending=False)
word_count_count.head(20)

In [ ]:
# wordcloud 的 input 是 dictionary
font_path = "./raw_data/SourceHanSansTW-Regular.otf"  # 中文字型路徑
wc_dict = dict(zip(word_count_count['word'], word_count_count['size']))
cloud = WordCloud(scale=4, max_words=200, background_color="white", font_path=font_path)
cloud.generate_from_frequencies(wc_dict)

plt.figure(figsize=(8, 4), dpi=150)
plt.imshow(cloud, interpolation="bilinear")
plt.axis("off")
plt.title("2 星飯店評論 — 正向情緒文字雲")
plt.show()

- **負面情緒**: 主要集中在客訴熱點 — 房間環境 (「髒」、「舊」、「吵」)、設備故障 (「壞」、「冷氣」)、服務態度 (「態度」、「差」、「等」)
- **正面情緒**: 即使在低分評論中,仍有少數正面表達,多半是「還算」、「勉強」等保留性讚美
- 可對應回 LDA 的主題分布,判斷飯店應優先改善哪些面向

從低分評論的內容可以發現,雖然整體評分偏低,但有時候客人會提到某些正面亮點 (例如地點方便、早餐不錯),
然而整體體驗因某一兩個嚴重缺點 (例如房間骯髒、櫃台態度差) 拉低分數。
這是飯店管理上值得注意的 — **單一嚴重缺失會壓倒多個優點**。